In [ ]:
# 1. Install the Hugging Face Hub library
!pip install -q huggingface_hub

# 2. Mount Drive and Download
import os
from google.colab import drive
from huggingface_hub import snapshot_download

# Mount Google Drive
drive.mount('/content/drive')

# Define your exact path
model_path = "/content/drive/MyDrive/1llm/Qn"
os.makedirs(model_path, exist_ok=True)

print(f"Downloading Qwen2.5-7B to {model_path}...\n(Make sure you have at least 15GB of free space in your Drive!)")

# Download the model directly to your Drive folder
snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir=model_path,
    ignore_patterns=["*.pt", "*.bin"] # Skips heavy legacy files to save Drive space
)

print("\n✅ Download complete! The model is now permanently saved in your Google Drive.")

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# Nuke any old, conflicting versions
!pip uninstall -y langchain langchain-community langchain-core langchain-huggingface smolagents gradio

# Install the 2026-stable AI stack
!pip install -U \
  langchain \
  langchain-community \
  langchain-core \
  langchain-huggingface \
  smolagents \
  transformers \
  bitsandbytes \
  accelerate \
  faster-whisper \
  edge-tts \
  gradio==3.50.2 \
  pytesseract \
  duckduckgo-search \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  soundfile
  ddgs

In [ ]:
# --- CELL 1: LOAD THE MODEL ---
import torch
from transformers import BitsAndBytesConfig
from smolagents import TransformersModel
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')
model_path = "/content/drive/MyDrive/1llm/Qn"

print("\nLoading the Brain (8-bit) into the T4 GPU... Please wait.")
quant_config = BitsAndBytesConfig(load_in_8bit=True)

agent_model = TransformersModel(
    model_id=model_path,
    model_kwargs={"quantization_config": quant_config},
    device_map="auto"
)
print("✅ Brain successfully loaded and resting in VRAM!")

In [ ]:
# =============================================================================
# ALLEN: DATA SCIENCE COMMAND CENTER – FULL FIXED CODE
# =============================================================================
# Run this cell after Cell 1 (which defines agent_model).

# -----------------------------------------------------------------------------
# 0. UPGRADE GRADIO & DEPENDENCIES (fixes "no attribute 'blocks'")
# -----------------------------------------------------------------------------
import subprocess
import sys

required_packages = ['gradio>=4.0', 'edge-tts', 'faster-whisper', 'pytesseract',
                     'langchain', 'langchain-community', 'langchain-huggingface',
                     'faiss-cpu', 'pillow', 'transformers']
for pkg in required_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All packages upgraded.")

# -----------------------------------------------------------------------------
# 1. IMPORTS
# -----------------------------------------------------------------------------
import os
import sqlite3
import asyncio
import tempfile
from pathlib import Path

import gradio as gr
import pytesseract
from PIL import Image
import numpy as np

from faster_whisper import WhisperModel
import edge_tts

from smolagents import CodeAgent, DuckDuckGoSearchTool, tool
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader

# -----------------------------------------------------------------------------
# 2. SQLITE PERMANENT MEMORY (on Google Drive)
# -----------------------------------------------------------------------------
DB_PATH = "/content/drive/MyDrive/allen_memory.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS chat_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  role TEXT,
                  content TEXT)''')
    conn.commit()
    conn.close()

def save_message(role, content):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("INSERT INTO chat_history (role, content) VALUES (?, ?)", (role, content))
    conn.commit()
    conn.close()

def load_history():
    if not os.path.exists(DB_PATH): return []
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT role, content FROM chat_history ORDER BY id ASC")
    rows = c.fetchall()
    conn.close()
    return [{"role": r[0], "content": r[1]} for r in rows]

def clear_memory():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    init_db()
    return []   # returns empty chat history for UI

# Initialize DB
init_db()

# -----------------------------------------------------------------------------
# 3. RAG & OCR TOOLS
# -----------------------------------------------------------------------------
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = None

@tool
def search_pdf(query: str) -> str:
    """Searches uploaded PDFs.

    Args:
        query (str): The search term to find within PDFs.

    Returns:
        str: The extracted content related to the query.
    """
    global vector_db
    if not vector_db: return "No PDFs uploaded."
    return "\n\n".join([d.page_content for d in vector_db.similarity_search(query, k=3)])

@tool
def read_image(image_path: str) -> str:
    """Extracts text from an image.

    Args:
        image_path (str): The path to the image file.

    Returns:
        str: The extracted text from the image.
    """
    try:
        return pytesseract.image_to_string(Image.open(image_path))
    except Exception as e:
        return f"OCR Error: {e}"

# -----------------------------------------------------------------------------
# 4. THE CORE LOGIC ENGINE (with tool toggles, camera, multi‑file)
# -----------------------------------------------------------------------------
async def chat_engine(txt_msg, mic_audio, file_msgs, camera_img,
                      history, use_think, use_search, use_pdf, use_ocr,
                      tts_voice, whisper_size):
    history = history or []
    user_text = txt_msg or ""

    # ---- Voice transcription ----
    if mic_audio:
        whisper_model = WhisperModel(whisper_size, device="cuda", compute_type="float16")
        segments, _ = whisper_model.transcribe(mic_audio)
        user_text += " " + "".join([s.text for s in segments])

    user_text = user_text.strip()
    display_text = user_text if user_text else "📎 [File/Photo Uploaded]"

    # ---- Process uploaded files ----
    global vector_db
    file_notes = ""

    # 2a. Regular file uploads (multiple)
    if file_msgs:
        for file_msg in file_msgs:
            ext = Path(file_msg.name).suffix.lower()
            if ext == '.pdf':
                pages = PyPDFLoader(file_msg.name).load_and_split()
                if vector_db is None:
                    vector_db = FAISS.from_documents(pages, embeddings)
                else:
                    vector_db.add_documents(pages)
                file_notes += f"[System: PDF {Path(file_msg.name).name} added to memory.]\n"
            elif ext in ('.png', '.jpg', '.jpeg', '.bmp', '.tiff'):
                file_notes += f"[System: Image {Path(file_msg.name).name} uploaded. Use OCR if enabled.]\n"
            else:
                file_notes += f"[System: Unsupported file type {ext}]\n"

    # 2b. Camera capture
    if camera_img is not None:
        temp_img = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        Image.fromarray(camera_img).save(temp_img.name)
        if use_ocr:
            ocr_text = pytesseract.image_to_string(Image.open(temp_img.name))
            file_notes += f"[OCR from camera: {ocr_text}]\n"
        else:
            file_notes += "[System: Camera photo captured (OCR disabled).]\n"

    if not user_text and not file_msgs and camera_img is None:
        return (gr.update(), gr.update(), gr.update(), gr.update(), history, None)

    # ---- Recent conversation context ----
    recent_context = "\n".join([f"{msg['role']}: {msg['content']}" for msg in history[-4:]])

    # ---- Build tool list from toggles ----
    tools = []
    if use_search:
        tools.append(DuckDuckGoSearchTool())
    if use_pdf and vector_db is not None:
        tools.append(search_pdf)
    if use_ocr:
        tools.append(read_image)

    # Recreate agent with selected tools (agent_model comes from Cell 1)
    agent = CodeAgent(
        tools=tools,
        model=agent_model,   # defined in Cell 1
        add_base_tools=True,
        max_steps=5
    )

    # ---- Execute agent ----
    prompt = f"""You are Allen, a Data Science AI assistant for Sri.
Recent Conversation Context:
{recent_context}

Current User Input: {user_text}
{file_notes}
CRITICAL: Wrap your final spoken text exactly in <code>final_answer('''...''')</code>"""

    try:
        if use_think:
            response = str(agent.run(prompt))
        else:
            response = "Deep Think is off. I need it to answer properly."
    except Exception as e:
        response = f"System Error: {e}"

    # ---- Save to SQLite ----
    save_message("user", display_text)
    save_message("assistant", response)

    # ---- Text-to-Speech ----
    tts_file = "reply.mp3"
    communicate = edge_tts.Communicate(response, tts_voice)
    await communicate.save(tts_file)

    # Update history
    history.append({"role": "user", "content": display_text})
    history.append({"role": "assistant", "content": response})

    # Return updates: clear inputs, return history and audio
    return (gr.update(value=""),          # clear text
            gr.update(value=None),        # clear mic
            gr.update(value=None),        # clear file input
            gr.update(value=None),        # clear camera
            history,
            tts_file)

# -----------------------------------------------------------------------------
# 5. UI – ALL-IN-ONE COMMAND CENTER WITH ICONS
# -----------------------------------------------------------------------------
print("🚀 Launching Allen Assistant...")

custom_css = """
.gradio-container { max-width: 1200px !important; margin: auto; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🧠 Allen: Data Science Command Center")

    # Load chat history from DB
    chat_state = gr.State(load_history())

    # ---- Row 1: Text + Mic ----
    with gr.Row():
        txt_input = gr.Textbox(placeholder="💬 Type a message...", show_label=False, scale=8)
        mic_input = gr.Audio(sources=["microphone"], type="filepath", show_label=False, scale=1, min_width=60)

    # ---- Row 2: File Upload + Camera ----
    with gr.Row():
        file_input = gr.File(label="📄 Upload Documents (PDF/Images)",
                             file_types=[".pdf", ".png", ".jpg", ".jpeg", ".bmp"],
                             file_count="multiple", scale=2)
        # FIX: Changed 'source' to 'sources' for webcam input in Gradio 4.0+
        camera_input = gr.Image(label="📸 Take a Photo", sources=["webcam"], type="numpy", scale=1)

    # ---- Row 3: Tool Toggles ----
    with gr.Row():
        think_toggle = gr.Checkbox(label="🧠 Think", value=True)
        search_toggle = gr.Checkbox(label="🌐 Search", value=True)
        pdf_toggle = gr.Checkbox(label="📄 PDF RAG", value=True)
        ocr_toggle = gr.Checkbox(label="🖼️ OCR", value=True)

    # ---- Row 4: Settings Dropdowns ----
    with gr.Row():
        tts_voice = gr.Dropdown(label="🗣️ TTS Voice",
                                choices=["en-GB-RyanNeural", "en-US-JennyNeural", "en-AU-NatashaNeural"],
                                value="en-GB-RyanNeural", scale=1)
        whisper_size = gr.Dropdown(label="🎤 Whisper Model",
                                   choices=["tiny", "base", "small", "medium", "large"],
                                   value="tiny", scale=1)

    # ---- Chat Display ----
    chatbot = gr.Chatbot(label="Live Conversation", height=400, value=chat_state.value)

    # ---- Hidden Audio Output ----
    voice_out = gr.Audio(visible=False, autoplay=True)

    # ---- Memory Management Buttons ----
    with gr.Row():
        clear_chat_btn = gr.Button("🧹 Clear Chat", variant="secondary")
        wipe_db_btn = gr.Button("🗑️ Wipe SQLite Memory", variant="stop")

    # -------------------------------------------------------------------------
    # EVENT HANDLERS
    # -------------------------------------------------------------------------
    chat_triggers = [txt_input.submit, mic_input.change, file_input.change, camera_input.change]
    for trigger in chat_triggers:
        trigger(
            fn=chat_engine,
            inputs=[txt_input, mic_input, file_input, camera_input, chatbot,
                    think_toggle, search_toggle, pdf_toggle, ocr_toggle,
                    tts_voice, whisper_size],
            outputs=[txt_input, mic_input, file_input, camera_input, chatbot, voice_out]
        )

    # Clear chat (UI only) – does not touch SQLite
    clear_chat_btn.click(lambda: [], outputs=[chatbot])

    # Wipe entire SQLite memory (clears DB and UI)
    wipe_db_btn.click(clear_memory, outputs=[chatbot])

# -----------------------------------------------------------------------------
# 6. LAUNCH
# -----------------------------------------------------------------------------
demo.launch(share=True, inline=False)